# Prototype 01 — Generate a Beam Database with Gemma on Kaggle

This notebook generates a **beam-search database** for the POSS-LOGIC-LM pipeline.

It uses a language model to produce multiple symbolic formalizations for each example:

```text
F1, F2, F3, F4, F5
```

Each candidate is saved with its beam-search score. These scores are later transformed into possibility degrees during the POSS-LOGIC-LM aggregation step.

Default configuration:
- Dataset: **PrOntoQA**
- Model: **Gemma-2-9B-Instruct**
- Number of candidates: `K = 5`
- Output file: `prontoqa_gemma.json`

## Before running on Kaggle

1. Enable a GPU accelerator: **Notebook settings → Accelerator → GPU T4/P100/A100**.
2. Add your Hugging Face token in Kaggle secrets:
   - Kaggle notebook → **Add-ons** → **Secrets**
   - Add a secret named `HF_TOKEN`
   - Paste your Hugging Face token as the value
3. Make sure you accepted the model license on Hugging Face if the selected model requires it.

## How to change the model

Edit only this variable:

```python
MODEL_NAME = "google/gemma-2-9b-it"
```

Examples:

```python
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
```

Then also update:

```python
OUTPUT_PATH = "prontoqa_qwen.json"
# or
OUTPUT_PATH = "prontoqa_llama3.json"
```

In [ ]:
!pip install -q transformers==4.45.2 datasets accelerate bitsandbytes huggingface_hub

## 1. Configuration

In [ ]:
import os
import gc
import json
import time
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# -----------------------------
# User configuration
# -----------------------------
MODEL_NAME = "google/gemma-2-9b-it"
DATASET_NAME = "renma/ProntoQA"
SPLIT_NAME = "validation"

BEAM_K = 5
MAX_NEW_TOKENS = 700
MAX_INPUT_TOKENS = 2048
MAX_EXAMPLES = 20   # Set to 500 for the full PrOntoQA run

OUTPUT_PATH = Path("prontoqa_gemma.json")

# -----------------------------
# Hugging Face authentication
# -----------------------------
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

print("Configuration loaded.")
print("Model:", MODEL_NAME)
print("Output:", OUTPUT_PATH)

## 2. Load model in 4-bit precision

This configuration is suitable for Kaggle GPUs with limited memory.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

print("Model loaded.")
print("Device:", model.device if hasattr(model, "device") else "auto")

## 3. Prompt template

The model must output a structured formalization containing:
- predicates;
- facts and rules;
- query.

In [ ]:
SYSTEM_MSG = """You are a formal logic translator for Prolog.

Return ONLY the following structure:

PREDICATES:
- predicate_name/arity

FACTS:
- fact_or_rule.

RULES:
- rule.

QUERY:
- query.

Rules:
- Use lowercase predicate names.
- Use underscores instead of spaces.
- Use Prolog-like Horn clauses when possible.
- Do not include markdown.
- Do not include explanations.
"""

def make_prompt(context: str, conclusion: str) -> str:
    return f"""{SYSTEM_MSG}

CONTEXT:
{context}

STATEMENT:
{conclusion}

OUTPUT:
"""

## 4. Beam generation function

The important part is `num_beams=BEAM_K`, `num_return_sequences=BEAM_K`, `return_dict_in_generate=True`, and `output_scores=True`.

The returned `sequences_scores` are the beam scores used later by POSS-LOGIC-LM.

In [ ]:
def generate_beam(context: str, conclusion: str, k: int = BEAM_K):
    gc.collect()
    torch.cuda.empty_cache()

    prompt = make_prompt(context, conclusion)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=k,
            num_return_sequences=k,
            do_sample=False,
            length_penalty=1.0,
            early_stopping=True,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    candidates = []
    for i, seq in enumerate(output.sequences):
        generated_ids = seq[prompt_len:]
        text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        score = None
        if hasattr(output, "sequences_scores") and output.sequences_scores is not None:
            score = float(output.sequences_scores[i].detach().cpu())

        candidates.append({
            "id": f"F{i+1}",
            "formalization": text,
            "log_score": score,
        })

    return candidates

## 5. Load and format PrOntoQA

This cell is specific to PrOntoQA.  
For another dataset, replace `format_example()`.

In [ ]:
ds = load_dataset(DATASET_NAME, split=SPLIT_NAME, token=hf_token)
dataset = list(ds)

print("Loaded examples:", len(dataset))
print("Available keys:", list(dataset[0].keys()))

def format_example(example, index: int):
    context = example.get("context", "")
    question = example.get("question", example.get("conclusion", ""))

    # PrOntoQA often stores the target statement inside the question field.
    if "?" in question:
        conclusion = question.split("?", 1)[-1].strip()
    else:
        conclusion = question.strip()

    label = example.get("answer", example.get("label", "Unknown"))
    label = str(label).strip()

    return {
        "id": example.get("id", f"pqa_{index:03d}"),
        "context": context,
        "conclusion": conclusion,
        "label": label,
    }

## 6. Generate the database

The notebook supports resume mode: if `OUTPUT_PATH` already exists, it skips examples already generated.

In [ ]:
def load_existing(output_path: Path):
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            rows = json.load(f)
        done = {row["id"] for row in rows}
        print(f"Resume mode: {len(rows)} examples already generated.")
        return rows, done
    return [], set()

def save_json(rows, output_path: Path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)

rows, done_ids = load_existing(OUTPUT_PATH)

start = time.time()
limit = min(MAX_EXAMPLES, len(dataset))

for i in range(limit):
    item = format_example(dataset[i], i)

    if item["id"] in done_ids:
        continue

    try:
        candidates = generate_beam(item["context"], item["conclusion"], k=BEAM_K)

        row = {
            "id": item["id"],
            "dataset": "PrOntoQA",
            "model": MODEL_NAME,
            "context": item["context"],
            "conclusion": item["conclusion"],
            "label": item["label"],
            "formalisations": [c["formalization"] for c in candidates],
            "log_scores": [c["log_score"] for c in candidates],
            "candidates": candidates,
        }

    except Exception as exc:
        row = {
            "id": item["id"],
            "dataset": "PrOntoQA",
            "model": MODEL_NAME,
            "context": item["context"],
            "conclusion": item["conclusion"],
            "label": item["label"],
            "error": repr(exc),
            "formalisations": [],
            "log_scores": [],
            "candidates": [],
        }

    rows.append(row)
    done_ids.add(item["id"])
    save_json(rows, OUTPUT_PATH)

    print(f"[{len(rows)}/{limit}] saved: {item['id']}")

print("Done.")
print("Elapsed seconds:", round(time.time() - start, 2))
print("Output file:", OUTPUT_PATH)

## 7. Quick inspection

In [ ]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    generated = json.load(f)

print("Generated examples:", len(generated))
print(json.dumps(generated[0], indent=2, ensure_ascii=False)[:2000])

## 8. Download the generated database

On Kaggle, the file is also available in the notebook output panel.

In [ ]:
print("Generated file:", OUTPUT_PATH)